# TabFM PyTorch CUDA Quickstart + Benchmark

This notebook validates CUDA availability, loads TabFM PyTorch checkpoints,
runs a multi-dataset benchmark, and stores benchmark artifacts under `artifacts/`.

In [ ]:
from pathlib import Path

from loguru import logger
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook but is not available.")

logger.info("CUDA available: {}", torch.cuda.is_available())
logger.info("CUDA device count: {}", torch.cuda.device_count())
logger.info("Current device: {}", torch.cuda.get_device_name(0))

## 1. Load TabFM classification and regression models

In [ ]:
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0

clf_model = tabfm_v1_0_0.load(model_type="classification", device="cuda")
reg_model = tabfm_v1_0_0.load(model_type="regression", device="cuda")

logger.info("Loaded model types: {}, {}", type(clf_model).__name__, type(reg_model).__name__)

## 2. Run multi-dataset benchmark

In [ ]:
import polars as pl

from tabfm_benchmark import render_summary, run_benchmark, save_results

results_df = run_benchmark(
    device="cuda",
    seed=42,
    n_estimators=32,
    use_ensemble_preset=True,
)
summary_df = render_summary(results_df)

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

save_results(results_df, artifacts_dir / "tabfm_benchmark_results.parquet")
save_results(summary_df, artifacts_dir / "tabfm_benchmark_summary.csv")

logger.info("Saved artifacts under {}", artifacts_dir.resolve())
results_df

## 3. Plot ranking views

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

pdf = results_df.filter(pl.col("status") == "ok").to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cls_df = pdf[pdf["task_type"] == "classification"].dropna(subset=["accuracy"])
if not cls_df.empty:
    cls_df = cls_df.sort_values("accuracy", ascending=False)
    sns.barplot(data=cls_df, x="accuracy", y="dataset", ax=axes[0], color="steelblue")
    axes[0].set_title("Classification Accuracy")
else:
    axes[0].set_title("Classification Accuracy (no successful runs)")

reg_df = pdf[pdf["task_type"] == "regression"].dropna(subset=["rmse"])
if not reg_df.empty:
    reg_df = reg_df.sort_values("rmse", ascending=True)
    sns.barplot(data=reg_df, x="rmse", y="dataset", ax=axes[1], color="darkorange")
    axes[1].set_title("Regression RMSE (lower is better)")
else:
    axes[1].set_title("Regression RMSE (no successful runs)")

plt.tight_layout()
plt.show()

## 4. License and usage reminder

TabFM checkpoint weights are distributed under a non-commercial license.
Use this notebook for research/evaluation unless you have separate commercial rights.